# Método de Trabalho

Bibliotecas utilizadas no desenvolvimento do código

In [ ]:
import pandas as pd

from langdetect import detect
from collections import Counter

from urlextract import URLExtract
import re

from nltk.corpus import words
import nltk
nltk.download('words')

import emoji
from emot.emo_unicode import EMOTICONS_EMO

import unicodedata

import contractions


from spellchecker import SpellChecker
from textblob import TextBlob

from sklearn.model_selection import train_test_split

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

from sentence_transformers import SentenceTransformer

import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

## Coleta dos Dados

In [ ]:
df = pd.read_csv('datasets/cyberbullying_tweets.csv')

df

## Pré-processamento dos Dados

### Amostragem representativa

### Limpeza

#### URLs

In [ ]:
# Remover URLs

extractor = URLExtract()

def remover_urls(texto):
    texto = str(texto)

    urls = extractor.find_urls(texto)

    for url in urls:
        texto = texto.replace(url, '')

    return texto

In [ ]:
# Verificar ocorrências de URLs

extractor = URLExtract()

def inspecionar_urls(texto):
    urls = []
    texto = str(texto)

    urls = extractor.find_urls(texto)

    if urls:
        print(urls)

df['tweet_text'].apply(inspecionar_urls)

#### E-mails

In [ ]:
# Remover e-mails

pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def remover_emails(texto):
    texto = str(texto)
    texto_limpo = pattern.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de e-mails

pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def inspecionar_emails(texto):
    texto = str(texto)
    emails = pattern.findall(texto)

    if emails:
        print(emails)

df['tweet_text'].apply(inspecionar_emails)

#### Menção a user

In [ ]:
# Remover menções a users

pattern = re.compile(r'(?:^| )(@[A-Za-z0-9_@]+)')

def remover_users(texto):
    texto = str(texto)
    texto_limpo = pattern.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de menções a users

pattern = re.compile(r'(?:^| )(@[A-Za-z0-9_]+)')

def inspecionar_users(texto):
    texto = str(texto)
    users = pattern.findall(texto)

    if users:
        print(users, texto)

df['tweet_text'].apply(inspecionar_users)

#### Espaços entre caracteres únicos consecutivos

In [ ]:
# Remover espaços entre caracteres únicos consecutivos

pattern = re.compile(r'(?:\b\w\s){3,}\w\b')

def remover_espacos(texto):
    texto = str(texto)
    texto_limpo = pattern.sub(lambda x: x.group().replace(' ', ''), texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de espaços entre caracteres únicos consecutivos

pattern = re.compile(r'(?:\b\w\s){3,}\w\b')

def inspecionar_espacos(texto):
    texto = str(texto)
    espaco = pattern.findall(texto)

    if espaco:
        print(texto)

df['tweet_text'].apply(inspecionar_espacos)

#### Múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return ' '.join(texto.split())

#### Filtrar idiomas

In [ ]:
# Inspecionar idiomas presentes no df

def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except:
        return 'desconhecido'

idiomas = df['tweet_text'].apply(detectar_idioma)
contagem = Counter(idiomas)

for idioma, quantidade in contagem.most_common():
    print(f'{idioma}: {quantidade} tweets ({quantidade/len(df)*100:.1f}%)')

In [ ]:
# Manter apenas dados da língua inglesa

def detectar_ingles(texto):
    try:
        return detect(str(texto)) == 'en'
    except:
        return False

#### Encapsulamento do pré-processamento 01

In [ ]:
def preprocessar_01(texto):
    texto = remover_urls(texto)
    texto = remover_emails(texto)
    texto = remover_users(texto)
    texto = remover_espacos(texto)
    texto = remover_multiplos_espacos(texto)

    return texto

df['tweet_text'] = df['tweet_text'].apply(preprocessar_01)
df = df[df['tweet_text'].apply(detectar_ingles)]

### Amostragem

In [ ]:
# Amostragem estratificada - 10% de cada grupo

df_rotulado, df_nao_rotulado = train_test_split(
    df,
    test_size = 0.9,
    stratify = df['cyberbullying_type'],
    random_state = 42
)

df_nao_rotulado_backup = df_nao_rotulado.copy() # Cria um backup dos dados não rotulados para fins de avaliação do kNN
df_nao_rotulado['cyberbullying_type'] = None # Retira os rótulos dos dados não rotulados para simular a situação real de um modelo kNN

print(f'Total: {len(df_rotulado)}')
print(df_rotulado['cyberbullying_type'].value_counts())

print(f'\nTotal: {len(df_nao_rotulado)}')
print(df_nao_rotulado['cyberbullying_type'].value_counts())

### Normalização

#### Normalização de caracteres alongados

In [ ]:
# Remover caracteres alongados

palavras_validas = set(words.words())
pattern = re.compile(r'(\w)\1{1,}')

def remover_caracteres_alongados(texto):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = pattern.sub(r'\1', token)
        resultado.append(token)

    return ' '.join(resultado)

In [ ]:
# Verificar ocorrências de caracteres alongados

pattern = re.compile(r'(\w)\1{2,}')

def inspecionar_caracteres_alongados(texto):
    texto = str(texto)
    espaco = pattern.findall(texto)

    if espaco:
        print(texto)

df_rotulado['tweet_text'].apply(inspecionar_caracteres_alongados)
df_nao_rotulado['tweet_text'].apply(inspecionar_caracteres_alongados)

#### Transformação de emojis e emoticons em texto

In [ ]:
# Converter emoticons em texto

def converter_emoticons(texto):
    texto = str(texto)

    for emoticon, significado in EMOTICONS_EMO.items():
        texto = texto.replace(emoticon, significado)
        
    return texto

In [ ]:
# Verificar ocorrências de emoticons

def inspecionar_emoticons(texto):
    texto = str(texto)
    encontrados = [emoticon for emoticon in EMOTICONS_EMO if emoticon in texto]
    
    if encontrados:
        print(texto)

df_rotulado['tweet_text'].apply(inspecionar_emoticons)
df_nao_rotulado['tweet_text'].apply(inspecionar_emoticons)

In [ ]:
# Converter emojis em texto

def converter_emojis(texto):
    return emoji.demojize(str(texto))

In [ ]:
# Verificar ocorrências de emojis

def inspecionar_emojis(texto):
    texto = str(texto)

    if emoji.emoji_count(texto) > 0:
        print(texto)

df_rotulado['tweet_text'].apply(inspecionar_emojis)
df_nao_rotulado['tweet_text'].apply(inspecionar_emojis)

#### Caracteres acentuados normalizados para o alfabeto inglês e em minúsculo

In [ ]:
# Remover acentos

def remover_acentos(texto):
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    
    return texto

In [ ]:
# Texto para minúsculo

def para_lowercase(texto):
    return str(texto).lower()

#### Conversão de gírias para palavras normais

In [ ]:
# Criar dicionário de gírias

def criar_girias(arquivo):
    dicionario_girias = {}

    df_girias = pd.read_csv(arquivo)
    dicionario_girias.update(dict(zip(df_girias['giria'].str.lower(), df_girias['significado'].str.lower())))

    return dicionario_girias

In [ ]:
# Converter gírias para palavras normais

def converter_girias(texto, dicionario_girias):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = dicionario_girias.get(token.lower(), token)
            
        resultado.append(token)
    
    return ' '.join(resultado)

In [ ]:
# Verificar ocorrências de gírias

dicionario_girias = criar_girias('datasets/girias.csv')

def inspecionar_girias(texto):
    texto = str(texto)
    tokens = texto.split()

    for token in tokens:
         if token.lower() not in palavras_validas:
               if dicionario_girias.get(token.lower()):
                    print(token)

df_rotulado['tweet_text'].apply(inspecionar_girias)
df_nao_rotulado['tweet_text'].apply(inspecionar_girias)

#### Expansão de contrações

In [ ]:
# Expandir contrações

def expandir_contracoes(texto):
    texto = str(texto)
    texto_expandido = contractions.fix(texto)
    
    return texto_expandido

In [ ]:
# Verificar ocorrências de contrações

lista_contracoes_reais = [
    chave for chave, valor in contractions.contractions_dict.items() 
    if chave.lower() != valor.lower()
]

regex_string = r'\b(' + '|'.join([re.escape(chave) for chave in lista_contracoes_reais]) + r')\b'
pattern_contracoes = re.compile(regex_string, flags=re.IGNORECASE)

def inspecionar_contracoes(texto):
    texto = str(texto)
    contracoes_encontradas = pattern_contracoes.findall(texto)
    
    if contracoes_encontradas:
        print(texto)

df_rotulado['tweet_text'].apply(inspecionar_contracoes)
df_nao_rotulado['tweet_text'].apply(inspecionar_contracoes)

#### Correção ortográfica

In [ ]:
# Corrigir erros ortográficos

def corrigir_ortografia(texto):
    texto = str(texto)
    texto_corrigido = str(TextBlob(texto).correct())
    
    return texto_corrigido

In [ ]:
# Verificar ocorrências de erros ortográficos

spell = SpellChecker(language='en')
palavra_pattern = re.compile(r'\b[a-zA-Z]+\b')

def inspecionar_ortografia(texto):
    cont = 0
    texto = str(texto)
    todas_palavras = palavra_pattern.findall(texto.lower())
    erros_encontrados = spell.unknown(todas_palavras)
    
    if erros_encontrados:
        print(f'{list(erros_encontrados)} em: {texto}')
        cont += 1

    return cont

cont = df_rotulado['tweet_text'].apply(inspecionar_ortografia)
print(f'Total de erros dados rotulados: {cont.sum()}')

cont = df_nao_rotulado['tweet_text'].apply(inspecionar_ortografia)
print(f'Total de erros dados não rotulados: {cont.sum()}')

#### Encapsulamento do pré-processamento 02

In [ ]:
arquivo = 'datasets/girias.csv'
palavras_validas = set(words.words())
dicionario_girias = criar_girias('datasets/girias.csv')

def preprocessar_02(texto, dicionario_girias):
    texto = remover_caracteres_alongados(texto)
    texto = converter_emoticons(texto)
    texto = converter_emojis(texto)
    texto = remover_acentos(texto)
    texto = para_lowercase(texto)
    texto = converter_girias(texto, dicionario_girias)
    texto = expandir_contracoes(texto)
    texto = corrigir_ortografia(texto)

    return texto

df_rotulado['tweet_text'] = df_rotulado['tweet_text'].apply(lambda x: preprocessar_02(x, dicionario_girias))
df_nao_rotulado['tweet_text'] = df_nao_rotulado['tweet_text'].apply(lambda x: preprocessar_02(x, dicionario_girias))

### Remoção e Lematização

#### Remoção de números

In [ ]:
# Remover números

pattern = re.compile(r'\d+')

def remover_numeros(texto):
    texto = str(texto)
    texto_limpo = pattern.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de números

pattern = re.compile(r'\d+')

def inspecionar_numeros(texto):
    texto = str(texto)
    numeros = pattern.findall(texto)

    if numeros:
        print(numeros, texto)

df_rotulado['tweet_text'].apply(inspecionar_numeros)
df_nao_rotulado['tweet_text'].apply(inspecionar_numeros)

#### Remoção de pontuações

In [ ]:
# Remover pontuações

pattern = re.compile(r'[^\w\s]')

def remover_pontuacoes(texto):
    texto = str(texto)
    texto_limpo = pattern.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de pontuações

pattern = re.compile(r'[^\w\s]')

def inspecionar_pontuacoes(texto):
    texto = str(texto)
    pontuacoes = pattern.findall(texto)

    if pontuacoes:
        print(pontuacoes, texto)

df_rotulado['tweet_text'].apply(inspecionar_pontuacoes)
df_nao_rotulado['tweet_text'].apply(inspecionar_pontuacoes)

#### Remoção de múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return ' '.join(texto.split())

#### Lematização

In [ ]:
# Salva uma cópia dos tweets originais para comparar com o df após a lematização

df_rotulado['tweet_original'] = df_rotulado['tweet_text']
df_nao_rotulado['tweet_original'] = df_nao_rotulado['tweet_text']

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lematizar(texto):
    texto = str(texto)
    tokens = nltk.word_tokenize(texto)
    tags = nltk.pos_tag(tokens)
    resultado = []
    
    for token, tag in tags:
        if tag in ('PRP', 'PRP$'):
            continue
        pos = get_wordnet_pos(tag)
        lema = lemmatizer.lemmatize(token, pos)

        if lema == 'be':
            continue
        
        resultado.append(lema)
    
    return ' '.join(resultado)

#### Encapsulamento do pré-processamento 03

In [ ]:
def preprocessar_03(texto):
    texto = remover_numeros(texto)
    texto = remover_pontuacoes(texto)
    texto = remover_multiplos_espacos(texto)
    texto = lematizar(texto)

    return texto

df_rotulado['tweet_text'] = df_rotulado['tweet_text'].apply(preprocessar_03)
df_nao_rotulado['tweet_text'] = df_nao_rotulado['tweet_text'].apply(preprocessar_03)

## Representação Vetorial dos Dados

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

tweets_rotulados = df_rotulado['tweet_text'].tolist()
embeddings_rotulados = model.encode(tweets_rotulados)

tweets_nao_rotulados = df_nao_rotulado['tweet_text'].tolist()
embeddings_nao_rotulados = model.encode(tweets_nao_rotulados)

print(f'Embeddings rotulados: {embeddings_rotulados.shape}')
print(f'Embeddings não rotulados: {embeddings_nao_rotulados.shape}')

## Armazenamento dos Dados

In [ ]:
try:
    conn = psycopg2.connect(
        host = 'localhost',
        database = 'CyberbullyingDetection',
        user = 'postgres',
        password = 'april2104'
    )

    cursor = conn.cursor()
    register_vector(conn)

    cursor.execute("TRUNCATE TABLE dados_rotulados RESTART IDENTITY;")
    cursor.execute("TRUNCATE TABLE dados_nao_rotulados RESTART IDENTITY;")

    rotulos = df_rotulado['cyberbullying_type'].tolist()
    registros_rotulados = list(zip(tweets_rotulados, embeddings_rotulados, rotulos))
    registros_nao_rotulados = list(zip(tweets_nao_rotulados, embeddings_nao_rotulados, df_nao_rotulado.index))

    execute_values(cursor, """
                INSERT INTO dados_rotulados (tweet, embedding, rotulo)
                VALUES %s
    """, registros_rotulados)

    execute_values(cursor, """
                INSERT INTO dados_nao_rotulados (tweet, embedding, indice_original)
                VALUES %s
    """, registros_nao_rotulados)

    conn.commit()
    print(f"{len(registros_rotulados) + len(registros_nao_rotulados)} registros inseridos com sucesso")

except Exception as e:
    conn.rollback()
    print(f"Erro: {e}")

finally:
    cursor.close()
    conn.close()

## Métricas de Avaliação

In [ ]:
try:
    conn = psycopg2.connect(
        host = 'localhost',
        database = 'CyberbullyingDetection',
        user = 'postgres',
        password = 'april2104'
    )

    cursor = conn.cursor()

    cursor.execute("SELECT id, tweet, rotulo, indice_original FROM dados_nao_rotulados")
    resultados = cursor.fetchall()

    df_resultado = pd.DataFrame(resultados, columns=['id', 'tweet', 'rotulo_knn', 'indice_original'])

except Exception as e:
    print(f"Erro: {e}")

finally:
    cursor.close()
    conn.close()

df_comparacao = df_resultado.merge(
    df_nao_rotulado_backup[['cyberbullying_type']],
    left_on = 'indice_original',
    right_index = True,
    how = 'left'
)

df_real = df_comparacao['cyberbullying_type']
df_pred = df_comparacao['rotulo_knn']

print(classification_report(df_real, df_pred))

#### Acurácia

In [ ]:
print(f"Acurácia:  {accuracy_score(df_real, df_pred):.4f}")

#### Precisão

In [ ]:
print(f"Precisão:  {precision_score(df_real, df_pred, average='weighted'):.4f}")

#### Recall

In [ ]:
print(f"Recall:    {recall_score(df_real, df_pred, average='weighted'):.4f}")

#### F1-score

In [ ]:
print(f"F1-Score:  {f1_score(df_real, df_pred, average='weighted'):.4f}")